## ------------------------------------------------------------ ###
## Set up environment
## ------------------------------------------------------------ ###

In [4]:
import os
import sys
import glob
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../../ProcessEvents/')
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR # MOLLY_DIR_FF, RAINFALL_CSV_DIR, ENSEMBLE_MEMBERS, , CATCHMENTS
event_details_fp = 'EventDetails_v4' # 'EventDetails_v5'
flood_5km_results_fp = "5km_total" # "5km_total_v5"

## ------------------------------------------------------------ ###
## Get dataframe with all catchment results
## ------------------------------------------------------------ ###

In [111]:
rainfall_events_all = []

for catchment_num in set(CATCHMENT_LOOKUP_DICT.keys()):
    print(catchment_num)
    if catchment_num in ['32']:
        continue
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    
    ## Get main event data
    fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_new_Nov.pkl")
    events_this_catchment = pd.read_pickle(fp)
    events_this_catchment['catchment_num'] = catchment_num
    events_this_catchment = events_this_catchment.reset_index(drop=True)
    
    ## Get temporal profile data
    tp_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/TemporalProfiles.csv"
    tp = pd.read_csv(tp_fp)
    tp = tp.reset_index(drop=True)
    
    ## Get landcover and topography data
    dst_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/DynamicSoilTings.csv"
    dst = pd.read_csv(dst_fp)
    del dst['event_num']
    dst = dst.reset_index(drop=True)

    events_this_catchment = events_this_catchment.loc[:, ~events_this_catchment.columns.duplicated()]
    tp = tp.loc[:, ~tp.columns.duplicated()]
    dst = dst.loc[:, ~dst.columns.duplicated()]    
    
    ## Join
    if not (len(tp) == len(events_this_catchment) == len(dst)):
        raise ValueError(f"Row counts differ: tp={len(tp)}, "f"events={len(events_this_catchment)}, dst={len(dst)}")
        continue
    this_catchment_joined = pd.concat([events_this_catchment, tp, dst],axis=1)
    
    # Add to big dataframe
    this_catchment_joined.reset_index(inplace=True, drop=True)
    rainfall_events_all.append(this_catchment_joined) 
    
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   
len(rainfall_events_all_df.loc[~rainfall_events_all_df["catchment_num"].map(lambda x: isinstance(x, (int, float))), "catchment_num"].unique())

33_a
15
19
34
102
61
71
3
82
37
28_c
86
23
70
40
74
66
25
78
27_a
80
32
107
97
27_c
65
47
52
62
28_a
51
6
18
83
63
73
90
7
76
22
53
29
17
57
56
72
24
89
92
95
36
75
11
69
54_a
77
9
27_b
20
1
93
5
96
84
67
8
105
60
44
91
48
54_b
30
28_b
59
106
39_b
12
87
58
13
64
41
38
43
45
39_c
16
88
31
14
4
46
104
33_b


FileNotFoundError: [Errno 2] No such file or directory: '/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/Catchment_33_b/GreatOuse_b_new_Nov.pkl'

### Join the landcover and topography stats

In [39]:
topo_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/topo_and_landcover_stats_for_unique_cells.pkl"
topo = pd.read_pickle(topo_fp)

In [44]:
cols = ["rainpeak_x_global", "rainpeak_y_global"]
rainfall_events_all_df["peak_cell_id"] = rainfall_events_all_df.groupby(cols, sort=False).ngroup()

In [46]:
merged = rainfall_events_all_df.merge(topo[['peak_cell_id']].drop_duplicates().assign(matched=True),
    on='peak_cell_id', how='right')

print("Unique topo peak_cell_ids:", topo['peak_cell_id'].nunique())
print("Matched topo peak_cell_ids:", check.loc[check['matched'].eq(True), 'peak_cell_id'].nunique())
print("Unmatched topo peak_cell_ids:", check['matched'].isna().sum())

Unique topo peak_cell_ids: 8929
Matched topo peak_cell_ids: 8929
Unmatched topo peak_cell_ids: 0


In [51]:
list(rainfall_events_all_df.columns)

['ens',
 'hydro_year',
 'event_num',
 'start_idx',
 'stop_idx',
 'max_precip_from_csv',
 'start_year',
 'start_month',
 'start_day',
 'start_hour',
 'event_durations',
 'time_mismatch',
 'rainpeak_found',
 'rainpeak_peak_value',
 'rainpeak_x_global',
 'rainpeak_y_global',
 'rainpeak_x_coord',
 'rainpeak_y_coord',
 'rainpeak_t_local',
 'rainpeak_y_idx',
 'rainpeak_x_idx',
 'rainpeak_rain_series',
 'rainpeak_max_precip',
 'rainpeak_total_precip',
 'rainpeak_no_wet_hours',
 'rainpeak_first_wet_idx',
 'rainpeak_trimmed_duration_hrs',
 'rainpeak_sm_day_before_onset',
 'rainpeak_10cm_area',
 'rainpeak_10cm_volume',
 'rainpeak_30cm_area',
 'rainpeak_30cm_volume',
 'floodpeak_found',
 'floodpeak_peak_value',
 'floodpeak_x_global',
 'floodpeak_y_global',
 'floodpeak_x_coord',
 'floodpeak_y_coord',
 'floodpeak_y_idx',
 'floodpeak_x_idx',
 'floodpeak_rain_series',
 'floodpeak_max_precip',
 'floodpeak_total_precip',
 'floodpeak_no_wet_hours',
 'floodpeak_first_wet_idx',
 'floodpeak_trimmed_duratio

In [50]:
topo.columns

Index(['peak_cell_id', 'peak_cell_slope_avg', 'peak_cell_slope_max',
       'peak_cell_elev_avg', 'peak_cell_elev_max', 'peak_cell_elev_min',
       'catchment_slope_avg', 'catchment_slope_max', 'catchment_elev_avg',
       'catchment_elev_max', 'catchment_elev_min', 'peak_cell_sink_frac',
       'peak_cell_outflow_N', 'peak_cell_outflow_S', 'peak_cell_outflow_E',
       'peak_cell_outflow_W', 'peak_cell_edge_truncated_N',
       'peak_cell_edge_truncated_S', 'peak_cell_edge_truncated_E',
       'peak_cell_edge_truncated_W', 'peak_cell_near_catchment_boundary',
       'peak_cell_urban_prop', 'peak_cell_inside_prop',
       'peak_cell_inside_water_prop', 'peak_cell_floodable_prop',
       'peak_cell_floodable_area_m2'],
      dtype='object')

## ------------------------------------------------------------ ###
## Check if any catchments have NANs for any variables
## ------------------------------------------------------------ ###

In [47]:
nulls = merged[merged['hc_at_peak'].isnull()]
np.unique(nulls)

KeyError: 'hc_at_peak'

In [14]:
nulls = rainfall_events_all_df[rainfall_events_all_df['fu_at_peak_new'].isnull()][['catchment_num', 'ens', 'fu_at_peak_new', 'lu_at_peak', 'event_num']]
nulls

,catchment_num,ens,fu_at_peak_new,lu_at_peak,event_num


## ------------------------------------------------------------ ###
## Check whether any grid cells have multiple events in them
## ------------------------------------------------------------ ###

In [15]:
dupes = rainfall_events_all_df[rainfall_events_all_df.duplicated(subset=["x_coord", "y_coord"],
        keep=False)].sort_values(["x_coord", "y_coord"])

len(dupes[['month', 'lu_at_peak', 'max_precip', 'ens', "x_coord", "y_coord"]])

106430

## ------------------------------------------------------------ ###
## Save to file
## ------------------------------------------------------------ ###

In [16]:
rainfall_events_all_df.to_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments.pkl")